In [1]:
import requests
import math

# =========================================
# 1) TOOLS
# =========================================
class WeatherTool:
    name = "weather_tool"

    def execute(self, city: str):
        url = f"https://wttr.in/{city}?format=j1"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        current = data["current_condition"][0]

        return {
            "city": city,
            "temperature_c": current["temp_C"],
            "description": current["weatherDesc"][0]["value"],
            "humidity": current["humidity"]
        }


class CalculatorTool:
    name = "calculator_tool"

    def execute(self, expression: str):
        # Very small safe calculator for demo
        allowed_names = {
            "abs": abs,
            "round": round,
            "min": min,
            "max": max,
            "pow": pow,
            "sqrt": math.sqrt
        }

        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return {
            "expression": expression,
            "result": result
        }


# =========================================
# 2) MCP-LIKE TOOL REGISTRY
# =========================================
class ToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, tool):
        self.tools[tool.name] = tool

    def list_tools(self):
        return list(self.tools.keys())

    def call_tool(self, tool_name: str, **kwargs):
        if tool_name not in self.tools:
            raise ValueError(f"Tool '{tool_name}' is not registered.")
        return self.tools[tool_name].execute(**kwargs)


# =========================================
# 3) AGENT
# =========================================
class SmartAgent:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def think(self, user_query: str):
        """
        Decide:
        1. Whether a tool is needed
        2. Which tool to use
        3. What inputs to pass
        """
        q = user_query.lower().strip()

        # Weather-related decision
        weather_keywords = ["weather", "umbrella", "rain", "temperature", "climate"]
        if any(word in q for word in weather_keywords):
            city = "Chennai"

            # Very simple city detection for demo
            known_cities = ["chennai", "bangalore", "mumbai", "delhi", "hyderabad"]
            for c in known_cities:
                if c in q:
                    city = c.title()
                    break

            return {
                "tool_needed": True,
                "tool_name": "weather_tool",
                "tool_args": {"city": city},
                "reason": "User is asking about weather-related information."
            }

        # Math-related decision
        math_keywords = ["calculate", "+", "-", "*", "/", "sqrt", "pow"]
        if any(word in q for word in math_keywords):
            expression = user_query.lower()
            expression = expression.replace("calculate", "").strip()

            return {
                "tool_needed": True,
                "tool_name": "calculator_tool",
                "tool_args": {"expression": expression},
                "reason": "User is asking for a calculation."
            }

        # No tool needed
        return {
            "tool_needed": False,
            "tool_name": None,
            "tool_args": {},
            "reason": "This question can be answered without external tools."
        }

    def analyze_result(self, decision: dict, result: dict):
        """
        Agent interprets tool output and gives final answer.
        """
        tool_name = decision["tool_name"]

        if tool_name == "weather_tool":
            desc = result["description"].lower()
            temp = int(result["temperature_c"])
            city = result["city"]

            if "rain" in desc or "drizzle" in desc or "thunder" in desc:
                return f"Yes, carry an umbrella. Current weather in {city} is {result['description']} and {temp}°C."
            elif temp >= 38:
                return f"No umbrella needed now, but it is very hot in {city} ({temp}°C). Carry water."
            else:
                return f"Weather in {city}: {result['description']}, {temp}°C. Umbrella may not be necessary now."

        if tool_name == "calculator_tool":
            return f"The result of {result['expression']} is {result['result']}."

        return "I used a tool successfully."

    def respond(self, user_query: str):
        print("USER QUERY:", user_query)
        print("-" * 60)

        # Step 1: Think
        decision = self.think(user_query)
        print("AGENT DECISION:")
        print(decision)
        print("-" * 60)

        # Step 2: Use tool if needed
        if decision["tool_needed"]:
            print(f"AGENT CALLING TOOL: {decision['tool_name']}")
            result = self.registry.call_tool(
                decision["tool_name"],
                **decision["tool_args"]
            )
            print("TOOL OUTPUT:")
            print(result)
            print("-" * 60)

            # Step 3: Analyze result
            final_answer = self.analyze_result(decision, result)
            print("FINAL ANSWER:")
            print(final_answer)
            return final_answer

        # No tool case
        final_answer = "I do not need any tool for this question."
        print("FINAL ANSWER:")
        print(final_answer)
        return final_answer


# =========================================
# 4) SETUP
# =========================================
registry = ToolRegistry()
registry.register(WeatherTool())
registry.register(CalculatorTool())

print("REGISTERED TOOLS:", registry.list_tools())
print("=" * 60)

agent = SmartAgent(registry)


# =========================================
# 5) TESTS
# =========================================
agent.respond("Should I carry an umbrella in Chennai today?")
print("\n" + "=" * 60 + "\n")

agent.respond("calculate 25 * 18")
print("\n" + "=" * 60 + "\n")

agent.respond("Hello, how are you?")

REGISTERED TOOLS: ['weather_tool', 'calculator_tool']
USER QUERY: Should I carry an umbrella in Chennai today?
------------------------------------------------------------
AGENT DECISION:
{'tool_needed': True, 'tool_name': 'weather_tool', 'tool_args': {'city': 'Chennai'}, 'reason': 'User is asking about weather-related information.'}
------------------------------------------------------------
AGENT CALLING TOOL: weather_tool
TOOL OUTPUT:
{'city': 'Chennai', 'temperature_c': '32', 'description': 'Sunny', 'humidity': '67'}
------------------------------------------------------------
FINAL ANSWER:
Weather in Chennai: Sunny, 32°C. Umbrella may not be necessary now.


USER QUERY: calculate 25 * 18
------------------------------------------------------------
AGENT DECISION:
{'tool_needed': True, 'tool_name': 'calculator_tool', 'tool_args': {'expression': '25 * 18'}, 'reason': 'User is asking for a calculation.'}
------------------------------------------------------------
AGENT CALLING TOOL

'I do not need any tool for this question.'